# Product Detection with YOLO26n

This notebook is a beginner-friendly product recognition tutorial. It starts with environment and YOLO26n model checks, then moves through dataset setup, CPU smoke training, validation, prediction, and troubleshooting.

The empty starter repo is safe to execute as a **scaffold checkpoint**: it can validate the project structure and then skip training, validation, and prediction until you add product images and matching YOLO label files under `data/products/`. After data exists, rerun the notebook so the strict preflight can unlock the CPU smoke training path.


## 1. What object detection means

Object detection finds **what** objects are in an image and **where** they are. The location is represented by a bounding box around each object. For product recognition, a detector might learn boxes for classes such as `product_a`, `product_b`, and `product_c`.

This is different from simple image classification: classification gives one label for the whole image, while detection can find multiple products in one image.

## 2. Why YOLO26n / nano

YOLO models are popular because they are designed for practical object detection workflows: prepare labeled images, train a model, validate it, and run predictions. The `n` in `yolo26n.pt` means **nano**, the smallest model size in this tutorial path.

We use nano first because it is the most beginner-friendly choice for a CPU-only smoke workflow. It will not guarantee high accuracy by itself, but it should be faster to load and test than larger model sizes.

## 3. CPU expectations

This tutorial assumes **CPU** execution. That keeps the workflow accessible on a normal laptop and avoids GPU-only setup steps. CPU checks can be slower, so this notebook starts with smoke checks only:

1. Import the `ultralytics` package.
2. Import the `YOLO` class.
3. Try to load `YOLO('yolo26n.pt')`.

If the model-load check fails, do not switch to another model automatically. The right next step is to verify that your installed Ultralytics version supports YOLO26 and that `yolo26n.pt` is available upstream.

## 4. Environment smoke check: import Ultralytics

Run this cell first. It confirms that Python can import the official Ultralytics package.

In [ ]:
import ultralytics

print(f"Ultralytics version: {ultralytics.__version__}")

## 5. Import the YOLO model class

The `YOLO` class is the main entry point used by Ultralytics examples for loading models. This cell should run before any model-load, training, or prediction code.

In [ ]:
from ultralytics import YOLO

print("YOLO import ok")

## 6. YOLO26n availability smoke check

This cell intentionally tries the exact model requested for the tutorial: `YOLO('yolo26n.pt')`.

If it fails, the notebook raises a clear error and stops. That is intentional: silently falling back to YOLO11 or another model would make the tutorial teach a different model than the one requested.

In [ ]:
try:
    model = YOLO('yolo26n.pt')
except Exception as exc:
    message = (
        "YOLO26n smoke check failed while loading yolo26n.pt.\n\n"
        "Do not switch this tutorial to YOLO11 automatically. "
        "Verify that your installed Ultralytics version supports YOLO26 "
        "and that yolo26n.pt is available upstream, then rerun this cell.\n\n"
        f"Original error: {type(exc).__name__}: {exc}"
    )
    raise RuntimeError(message) from exc

print(f"Loaded yolo26n.pt for this CPU tutorial: {type(model).__name__}")
print("Environment and model availability checks complete. Continue to the dataset scaffold checkpoint.")


## 7. Collect images for a tiny starter dataset

Start with a tiny dataset that is easy to collect and easy to inspect. For this tutorial, use the placeholder classes `product_a`, `product_b`, and `product_c` instead of real product names. That keeps the lesson focused on the workflow, not on a specific brand or store catalog.

A beginner-friendly collection plan looks like this:

- collect a few clear photos for each placeholder class
- keep most images for `train` and hold a smaller set back for `val`
- avoid duplicate or near-identical photos, because they teach the model less than varied views
- vary lighting, camera distance, angle, shelf position, and background
- include packaging changes, dents, reflections, and partial occlusion
- take some images with more than one product in the frame so you practice multi-box labeling later

A simple starter split is 5 training images and 2 validation images per class. That is only a smoke dataset, but it is enough to check that the folder layout and annotation flow work.

Use `train` images for learning and `val` images for the held-out check. Do not copy the same photo into both folders, because that makes the validation result look better than it really is.

## 8. Annotate boxes in Roboflow or by hand

Roboflow is optional here. It is handy for drawing boxes and exporting YOLO labels, but this tutorial does not require an API key or a paid account. If you use it, export in YOLO detection format. The exported files must match the same folder layout used by the local fallback path below.

```text
data/products/
  images/
    train/
    val/
  labels/
    train/
    val/
```

The notebook and dataset YAML expect that layout, so exported Roboflow folders should be copied into `data/products/` without changing the `images` and `labels` structure. The class order must also stay aligned with `data/product_dataset.yaml`, where `product_a` is class 0, `product_b` is class 1, and `product_c` is class 2.

If you do not use Roboflow, you can create the same layout by hand and write the YOLO `.txt` files yourself. That local fallback is the default path for this tutorial.

## 9. Common annotation mistakes to avoid

Watch for these edge cases while you label images:

- empty label file for a background or no-object image, which is valid
- missing image file for a label file
- missing label file for an image that should have boxes
- class order mismatch between your labels and `data/product_dataset.yaml`
- multiple products in the same photo, which need one row per box
- different product packaging, which should still use the same class when it is the same product category
- boxes that are too loose, too tight, or drawn around several products at once
- pixel coordinates instead of normalized YOLO coordinates

If an image has nothing to label, keep the matching `.txt` file but leave it empty. If one photo contains several products, add one row per object. Those two cases are normal and help the tutorial stay consistent with the local fallback layout.

## 10. Preflight: scaffold checkpoint or strict dataset validation

Before you train, run the local validator against the same dataset YAML the model will use: `data/product_dataset.yaml`. This catches missing folders, class-order mistakes, image/label pairing problems, and malformed YOLO label rows before Ultralytics starts a slower CPU run.

The empty starter repo uses a scaffold checkpoint with `--allow-empty`; that proves the YAML and folder rules are understandable, but it is **not** proof that training data is ready. When train and val image folders contain files, the notebook automatically switches to strict validation without `--allow-empty`. If strict validation fails, fix the dataset instead of skipping the error.


In [ ]:
import subprocess
import sys
from pathlib import Path

image_suffixes = {'.bmp', '.jpeg', '.jpg', '.png', '.tif', '.tiff', '.webp'}
train_image_dir = Path('data/products/images/train')
val_image_dir = Path('data/products/images/val')

def directory_has_images(directory):
    return directory.exists() and any(
        path.is_file() and path.suffix.lower() in image_suffixes
        for path in directory.rglob('*')
    )

SCAFFOLD_MODE = not (directory_has_images(train_image_dir) and directory_has_images(val_image_dir))
DATASET_READY_FOR_TRAINING = not SCAFFOLD_MODE

preflight_command = [
    sys.executable,
    'scripts/validate_yolo_dataset.py',
    '--data',
    'data/product_dataset.yaml',
]

if SCAFFOLD_MODE:
    preflight_command.append('--allow-empty')

print('Running dataset preflight:', ' '.join(preflight_command))
subprocess.run(preflight_command, check=True)

if SCAFFOLD_MODE:
    print('Scaffold checkpoint passed. Add train and val images with matching labels before running training, validation, or prediction.')
else:
    print('Strict dataset preflight passed. Continue to the CPU smoke training cell.')


## 11. CPU-safe training smoke run

This training cell is intentionally tiny: `epochs=1`, `imgsz=320`, and `device='cpu'`. It teaches the training workflow without requiring a GPU. On a laptop CPU it may still take a while, but it should be much safer than a long run such as 100 epochs.

Treat this as a **smoke/learning run**, not a useful product detector. A tiny MVP dataset can prove that paths, labels, and training code connect correctly, but it cannot promise useful accuracy. In scaffold mode this cell prints a skip message instead of pretending training succeeded. Real quality needs more varied images, careful labels, and repeated validation after this tutorial step.


In [ ]:
training_config = dict(
    model='yolo26n.pt',
    data='data/product_dataset.yaml',
    epochs=1,
    imgsz=320,
    device='cpu',
)

if not DATASET_READY_FOR_TRAINING:
    training_model = None
    train_results = None
    print('Skipping CPU smoke training because this is still the empty scaffold checkpoint.')
else:
    training_model = YOLO(training_config['model'])
    train_results = training_model.train(
        data=training_config['data'],
        epochs=training_config['epochs'],
        imgsz=training_config['imgsz'],
        device=training_config['device'],
        project='runs/detect',
        name='train_cpu_smoke',
        exist_ok=True,
    )
    print('CPU smoke training finished. Check runs/detect/train_cpu_smoke for logs and weights.')

train_results


## 12. Validate the trained model

Validation checks the model against the held-out `val` images from `data/product_dataset.yaml` after a training run has actually happened. In scaffold mode this section is skipped because there is no trained checkpoint or real validation set yet. The goal in this beginner lesson is to learn how to read the output, not to claim strong accuracy from a smoke dataset.

Beginner metric guide:

- **Precision** asks: when the model predicts a product box, how often is that prediction correct? Low precision often means false alarms or boxes assigned to the wrong product.
- **Recall** asks: of the real products in the validation images, how many did the model find? Low recall often means the model missed products.
- **mAP** summarizes detection quality across confidence thresholds. On a tiny smoke dataset it can swing wildly, so use it only as a workflow signal here.
- A **confusion matrix** shows when one class is mistaken for another, such as `product_a` being confused with `product_b`. Similar packaging, class-order mistakes, or inconsistent boxes can all create confusion between classes.


In [ ]:
if train_results is None:
    validation_results = None
    print('Skipping validation because training did not run in scaffold mode.')
else:
    validation_results = training_model.val(
        data='data/product_dataset.yaml',
        imgsz=320,
        device='cpu',
        project='runs/detect',
        name='val_cpu_smoke',
        exist_ok=True,
    )
    print('Validation finished. Read the precision, recall, mAP, and confusion matrix outputs as learning signals only.')

validation_results


## 13. Make a prediction with trained weights

This lesson uses the best checkpoint when training has already produced one. The notebook looks for the newest `runs/detect/train*/weights/best.pt` file first, because that is the normal place Ultralytics saves the best model from a training run.

If no trained checkpoint exists yet but validation images are present, the lesson can fall back to `yolo26n.pt` as a CPU smoke prediction so you can learn the prediction flow. In the empty scaffold, prediction is skipped because there is no local image source to run against.


In [ ]:
from pathlib import Path

val_source = Path('data/products/images/val')
has_prediction_images = val_source.exists() and any(
    path.is_file() and path.suffix.lower() in image_suffixes
    for path in val_source.rglob('*')
)

if not has_prediction_images:
    prediction_results = None
    print('Skipping prediction because data/products/images/val has no images yet.')
else:
    predict_candidates = sorted(
        Path('runs/detect').glob('train*/weights/best.pt'),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    weights_path = str(predict_candidates[0]) if predict_candidates else 'yolo26n.pt'

    print(f'Using weights for prediction: {weights_path}')
    predict_model = YOLO(weights_path)
    prediction_results = predict_model.predict(
        source=str(val_source),
        imgsz=320,
        device='cpu',
        project='runs/detect',
        name='predict_cpu_smoke',
        exist_ok=True,
    )
    print('Prediction finished. Look for saved outputs under runs/detect/predict_cpu_smoke.')

prediction_results


## 14. Troubleshooting the first prediction run

If prediction does not behave the way you expect, check these common beginner issues:

- **Missing dataset files**: confirm `data/product_dataset.yaml` points to `data/products`, and check that `images/val` exists with at least one test image.
- **Malformed labels**: run the dataset validator again if boxes are missing, class IDs are wrong, or coordinates are not normalized between 0 and 1.
- **CPU slowness**: keep `imgsz=320`, use a small val folder, and expect a slower first pass on a laptop CPU.
- **Similar-looking products**: add more variety in lighting, angles, and packaging so the model sees clearer differences.
- **Packaging changes**: treat the same product family as the same class unless the class definition itself changes.
- **Class confusion**: if `product_a` and `product_b` get mixed up, check class order, box quality, and whether the training set has enough examples of each class.

For this tutorial, prediction errors are learning signals. They usually mean the dataset needs more variety or cleaner labels, not that the notebook workflow is wrong.

## 15. Advanced note: YOLO26 inference behavior

YOLO26 defaults to end-to-end, NMS-free inference. That is an advanced implementation detail, so this MVP does not add manual NMS code or export/NMS tuning steps. Keep the tutorial focused on the notebook workflow first; export and deployment tuning can wait for later lessons.

## 16. What comes next

You now have one beginner-friendly path for environment checks, scaffold dataset validation, dataset setup, CPU smoke training, validation, prediction, troubleshooting, and repo-level launcher guidance. Keep using scaffold mode until your local `data/products/` folders contain real train and val images with matching YOLO label files, then rerun the notebook and let the strict preflight decide whether training can start.
